In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from matplotlib.colors import LinearSegmentedColormap
import torch
import torch.nn as nn
from pathlib import Path
from datetime import datetime

import sys
sys.path.append("../../tools")
import architec as acht
import evaluation as eva
import prepare as prep
import functions as func

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [2]:
import importlib
#importlib.reload(func)


In [7]:
rep = 3
load_dir1 = Path(f"../../saved_models/swi_species_3fold/sp10_SS_CNN1D_r{rep}")

smodel = func.load_saved_model_SS_CNN1D(load_dir1, device)
sevaluator = func.TrteEvaluator(smodel)

#-------------- Set up test set -------------------------
# #overlap_te = 'Ver1.9/ver1.9.6_upto20_te/10-20_96000'
# overlap_te = 'Ver1.9/ver1.9.6_upto20_te/20-20_102400'
# df_te = pd.read_csv(f'../../Databases/{overlap_te}.csv')
# narray_te = df_te.iloc[:, :-2].values

# folder_te = "../../Databases/Ver1.9/ver1.9.6_upto20_te"
# narray_te, used_files = prep.load_training_dataset_from_folder(folder_te, sample_size=2000, random_state=42)

#split_df = pd.read_csv(f"split_MD_v0/split_MD_{rep}.csv")
#split_df = pd.read_csv(f"split_MD_v1/split_MD_{rep}.csv")
split_df = pd.read_csv(f"../Fig2/split_ver1.9.6_upto20_v2/split_{rep}.csv")

#folder_path = "../../Databases/Ver1.9/ver1.9.6_upto20"
folder_path = "../../Databases/Ver1.9/ver1.9.6_upto20_v2"


In [ ]:
rich = 10
narray_te = prep.load_split_dataset(
    split_df,
    folder_path,
    richness_list=[rich],
    subset="te"
)

# Evaluate the model on the test set
y_te, y_prob = sevaluator.predict_array_SS(narray_te, device, sample_size=None)

threshold = 0.3
metrics_list = []

for i in range(rich):
    metrics_i = eva.eval_binary(y_te[:, i], y_prob[:, i], threshold=threshold)
    metrics_i["Species"] = i
    metrics_list.append(metrics_i)

df_metrics_species = pd.DataFrame(metrics_list)

metrics = (
    df_metrics_species
    .drop(columns=["Species"])
    .mean().to_dict()
)

metrics["test_richness"] = rich
display(metrics)


te shape = (19648, 465)


{'Accuracy': 0.6616703990228012,
 'Balanced_Acc': 0.6942332478950693,
 'F1': 0.5855712644425656,
 'MCC': 0.35862845365685836,
 'AUC': 0.7673979649390638,
 'AUPRC': 0.5729185160868816,
 'AUPRC_rel': 0.8657994375967053,
 'Brier': 0.17304082985086738,
 'pos_rate': 0.3070541530944625,
 'test_richness': 10}